# Supervised Learning Project on Heart Failure Prediction with KNN

### 1. Introduction & Setup

This project aims to build a K-Nearest Neighbors (KNN) classifier to predict mortality in patients with heart failure. The model will be trained on the "Heart Failure Prediction" dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

### 2. Data Preparation

In [16]:
file_path = '../data/heart_disease_uci.csv' 
df = pd.read_csv(file_path)

In [18]:
print("First 5 rows of the dataset:")
print(df.head())

print("\nDataset Information:")
df.info()

print("\nDescriptive Statistics:")
print(df.describe())
print(f"\nDataset shape: {df.shape}")


First 5 rows of the dataset:
   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4      

###  Data Cleaning and Preprocessing

In [ ]:
df_cleaned = df.drop(['id', 'dataset'], axis=1)

df_cleaned['target'] = (pd.to_numeric(df_cleaned['num'], errors='coerce') > 0).astype(int)
df_cleaned = df_cleaned.drop('num', axis=1)

In [ ]:
X = df_cleaned.drop('target', axis=1)
y = df_cleaned['target']

numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')

X_processed = pd.get_dummies(X, columns=categorical_cols, dummy_na=True, drop_first=True)

print("\nShape of features after one-hot encoding:", X_processed.shape)
print("Columns created:", X_processed.columns)
print("\nData head after all processing:")
print(X_processed.head())


Shape of features after one-hot encoding: (920, 25)
Columns created: Index(['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'sex_Male',
       'sex_nan', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina',
       'cp_nan', 'fbs_True', 'fbs_nan', 'restecg_normal',
       'restecg_st-t abnormality', 'restecg_nan', 'exang_True', 'exang_nan',
       'slope_flat', 'slope_upsloping', 'slope_nan', 'thal_normal',
       'thal_reversable defect', 'thal_nan'],
      dtype='object')

Data head after all processing:
   age  trestbps   chol  thalch  oldpeak   ca  sex_Male  sex_nan  \
0   63     145.0  233.0   150.0      2.3  0.0      True    False   
1   67     160.0  286.0   108.0      1.5  3.0      True    False   
2   67     120.0  229.0   129.0      2.6  2.0      True    False   
3   37     130.0  250.0   187.0      3.5  0.0      True    False   
4   41     130.0  204.0   172.0      1.4  0.0     False    False   

   cp_atypical angina  cp_non-anginal  ...  restecg_st-t abnormal